# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font>  — Avance 7</center>

## **<font color="#0036a3">Avance 7 — Reconstrucción 3D a partir de la profundidad estimada</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10 · Equipo 52</font>**

---

Este avance cierra el pipeline: a partir del **mapa de profundidad** que predice **MonoIIT** (el modelo de menor error del estudio), reconstruimos la **geometría 3D** de la escena endoscópica como una **nube de puntos** y la comparamos con el *ground truth* del sensor estéreo de SCARED.

## 1. Configuración del entorno, modelo MonoIIT y enhancements

Mismo setup que el Avance 6: rutas (Colab/local), carga del *split* oficial (550 frames), del modelo **MonoIIT** y de los métodos de realce. Reejecutar tal cual.

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab; IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT  = BASE / "scared_raw"
    EDAM_PATH    = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH  = BASE / "EndoLMSPEC"
    IAT_PATH     = BASE / "EndoViT"
    MONOVIT_PATH = BASE / "MonoViT"
    STTN_PATH    = BASE / "Endo-STTN"
    W            = BASE / "scared weights"
    REPO_ROOT    = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git","clone","--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git", str(REPO_ROOT)])
    else:
        subprocess.check_call(["git","-C",str(REPO_ROOT),"fetch","origin"])
        subprocess.check_call(["git","-C",str(REPO_ROOT),"reset","--hard","origin/main"])
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
else:
    BASE         = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT  = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH    = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH  = Path("E:/EndoLMSPEC")
    IAT_PATH     = Path("E:/EndoVit")
    MONOVIT_PATH = Path("E:/MonoViT")
    STTN_PATH    = Path("E:/Endo-STTN")
    W            = BASE
    REPO_ROOT    = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    SPLIT_FILE   = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"

W_MONOIIT      = W / "monoIIT_weights" / "trained-winner-weights"
LMSPEC_WEIGHTS = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS    = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
STTN_WEIGHTS   = STTN_PATH / "release_model" / "pretrained_model" / "gen_00009.pth"
NPZ_CACHE      = BASE / "split_frames.npz"

def load_split(sf):
    items=[]
    with open(sf) as f:
        for line in f:
            line=line.strip()
            if not line: continue
            folder, fid, _ = line.split()
            ds, kf = folder.split("/")
            items.append(("dataset_"+ds.replace("dataset",""), "keyframe_"+kf.replace("keyframe",""), int(fid)))
    return items
SPLIT_ITEMS = load_split(SPLIT_FILE)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds,kf,fid in SPLIT_ITEMS: SPLIT_BY_KF[(ds,kf)].append(fid)
print(f"{'Colab' if IN_COLAB else 'Local'} | split {len(SPLIT_ITEMS)} frames")

In [ ]:
import numpy as np
# Cargar imagenes + GT del npz cacheado
def _key(ds,kf,fid): return f"{ds}|{kf}|{fid}"
SPLIT_DATA = {}
_npz = np.load(NPZ_CACHE, allow_pickle=True)
for ds,kf,fid in SPLIT_ITEMS:
    k=_key(ds,kf,fid); ik,gk="img_"+k,"gt_"+k
    if ik in _npz.files:
        gt=_npz[gk] if gk in _npz.files else None
        if gt is not None and gt.size==1 and np.isnan(gt).all(): gt=None
        SPLIT_DATA[k]=(_npz[ik], gt)
def load_split_frame(ds,kf,fid):
    return SPLIT_DATA.get(_key(ds,kf,fid),(None,None))
print(f"Frames en memoria: {len(SPLIT_DATA)}")

In [ ]:
import torch, importlib.util as _ilu, types
import torch.nn as nn
import numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_NETDIR = MONOVIT_PATH / "networks"
for _k in list(sys.modules):
    if _k=="networks" or _k.startswith("networks."): del sys.modules[_k]
_pkg=types.ModuleType("networks"); _pkg.__path__=[str(_NETDIR)]; sys.modules["networks"]=_pkg
def _ls(name,fn):
    sp=_ilu.spec_from_file_location(f"networks.{name}",str(_NETDIR/fn))
    m=_ilu.module_from_spec(sp); sys.modules[f"networks.{name}"]=m; sp.loader.exec_module(m); setattr(_pkg,name,m); return m
_ls("hr_layers","hr_layers.py")
_hr = _ls("hr_decoder","hr_decoder.py")
mpvit_small=_ls("mpvit","mpvit.py").mpvit_small
DepthDecoderHR = _hr.DepthDecoder

# MonoIIT usa encoder mpvit + decoder HR-Depth OFICIAL (convs.f4, X_00, attention),
# igual que MonoViT. Se carga con DepthDecoderHR() defaults (como evaluate_depth.py oficial).
enc = mpvit_small(); enc.num_ch_enc=[64,128,216,288,288]
_ed = torch.load(W_MONOIIT/"encoder.pth", map_location=DEVICE)
MH, MW = _ed.get("height",192), _ed.get("width",640)
enc.load_state_dict({k:v for k,v in _ed.items() if k in enc.state_dict()}); enc.to(DEVICE).eval()

_sd = torch.load(W_MONOIIT/"depth.pth", map_location=DEVICE)
dec = DepthDecoderHR()
_r = dec.load_state_dict(_sd, strict=False); dec.to(DEVICE).eval()
print(f"MonoIIT cargado {MH}x{MW} (HR-Depth) | missing={len(_r.missing_keys)} unexpected={len(_r.unexpected_keys)}")

import cv2, PIL.Image as pil
from torchvision import transforms
def predict_depth(img, max_depth=150.0, min_depth=0.1):
    H,W_=img.shape[:2]
    t=transforms.ToTensor()(pil.fromarray(img).resize((MW,MH),pil.LANCZOS)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): out=dec(enc(t))
    disp=out[("disp",0)].squeeze().detach().cpu().numpy()
    sd=(1.0/max_depth)+((1.0/min_depth)-(1.0/max_depth))*disp
    sd=cv2.resize(sd,(W_,H)); return 1.0/sd

In [ ]:
# Cargar los enhancements (mismo codigo que el notebook principal)
import torchvision.transforms as T
subprocess.check_call([sys.executable,"-m","pip","install","-q","IQA_pytorch","path"])

def _load_endolmspec(p, device):
    _orig=sys.path.copy()
    clean=[str(p)]+[x for x in sys.path if "EndoSLAM" not in x and "endosfm" not in x.lower()
                    and "HADepth" not in x and "EndoViT" not in x and "EndoVit" not in x]
    for k in list(sys.modules):
        if k in ("utils","generator","unet") or k.startswith("utils."): del sys.modules[k]
    try:
        sys.path=clean
        sp=_ilu.spec_from_file_location("generator", p/"generator.py")
        mod=_ilu.module_from_spec(sp); sp.loader.exec_module(mod); G=mod.Generator
    finally: sys.path=_orig
    return G(n_channels=3, device=device, bilinear=False)
lmspec_net=_load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE)); lmspec_net.to(DEVICE).eval()

for k in list(sys.modules):
    if k=="utils" or k.startswith("utils."): del sys.modules[k]
sys.modules["imp"]=types.ModuleType("imp")
_sp=_ilu.spec_from_file_location("IAT_main_a5", IAT_PATH/"experiments"/"model"/"IAT_main.py")
_im=_ilu.module_from_spec(_sp)
if str(IAT_PATH/"experiments") not in sys.path: sys.path.insert(0,str(IAT_PATH/"experiments"))
_sp.loader.exec_module(_im)
iat_net=_im.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE)); iat_net.to(DEVICE).eval()

def correct_none(img): return img
def correct_retinex(img, sigma=30):
    f=img.astype(np.float32)+1.0; r=np.zeros_like(f)
    for c in range(3):
        b=cv2.GaussianBlur(f[:,:,c],(0,0),sigma); r[:,:,c]=np.log(f[:,:,c])-np.log(b+1.0)
    r-=r.min(); return (r/(r.max()+1e-8)*255).astype(np.uint8)
def correct_endolmspec(img):
    t=T.ToTensor()(img).to(DEVICE)
    with torch.no_grad(): _,o=lmspec_net(t)
    return (o["subnet_16"][0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)
def correct_iat(img):
    t=torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _,_,e=iat_net(t)
    return (e[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

CORRECTIONS={"none":correct_none,  # "retinex":correct_retinex,  # desactivado a peticion del profesor (conservado por si acaso)
             "endolmspec":correct_endolmspec,"iat":correct_iat}
print("Enhancements:", list(CORRECTIONS.keys()))
print("(endosttn se omite en esta visualizacion por ser temporal/pesado; se puede anadir si se desea)")

In [ ]:
# --- Endo-STTN: inpainting temporal de especularidades (mismo codigo que Avance 5) ---
STTN_CKPT_DIR    = STTN_PATH / "release_model" / "pretrained_model"
STTN_GDRIVE_ID   = "14sdaDejsxgRuzHBSuqH2xEpbxuqyWI-R"
if IN_COLAB and not STTN_WEIGHTS.exists():
    STTN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable,"-m","pip","install","-q","gdown"])
    import gdown; gdown.download(id=STTN_GDRIVE_ID, output=str(STTN_WEIGHTS), quiet=False)

def _load_endo_sttn(sttn_path, ckpt, device):
    _orig_path = sys.path.copy()
    _orig_mods = {k: sys.modules[k] for k in list(sys.modules)
                  if k=="core" or k.startswith("core.") or k=="model" or k.startswith("model.")}
    for k in list(sys.modules):
        if k=="core" or k.startswith("core.") or k=="model" or k.startswith("model."): del sys.modules[k]
    try:
        sys.path.insert(0, str(sttn_path))
        net_mod = __import__("model.sttn", fromlist=["InpaintGenerator"])
        utils_mod = __import__("core.utils", fromlist=["Stack","ToTorchFormatTensor"])
        model = net_mod.InpaintGenerator().to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device)["netG"]); model.eval()
        Stack = utils_mod.Stack; ToTorch = utils_mod.ToTorchFormatTensor
    finally:
        sys.path = _orig_path
        for k in list(sys.modules):
            if k=="core" or k.startswith("core.") or k=="model" or k.startswith("model."): del sys.modules[k]
        sys.modules.update(_orig_mods)
    return model, Stack, ToTorch

STTN_W, STTN_H = 288, 288
STTN_REF_LEN, STTN_STRIDE = 10, 5
_sttn_ok = STTN_WEIGHTS.exists()
if _sttn_ok:
    sttn_model, _Stack, _ToTorch = _load_endo_sttn(STTN_PATH, STTN_WEIGHTS, DEVICE)
    _sttn_to_tensors = T.Compose([_Stack(), _ToTorch()])
    print("Endo-STTN OK")
else:
    print("Endo-STTN: pesos no encontrados; se omitira esa columna")

def _sttn_specular_mask(img_rgb, dil=8):
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L >= np.percentile(L, 97)).astype(np.uint8)
    if dil: m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil)), iterations=1)
    return m

def _sttn_get_ref_index(nb_ids, length):
    return [i for i in range(0, length, STTN_REF_LEN) if i not in nb_ids]

@torch.no_grad()
def endo_sttn_inpaint_sequence(frames_rgb):
    if not _sttn_ok: return frames_rgb
    H0, W0 = frames_rgb[0].shape[:2]
    pil_frames = [pil.fromarray(f).resize((STTN_W,STTN_H), pil.LANCZOS) for f in frames_rgb]
    masks_np   = [cv2.resize(_sttn_specular_mask(f),(STTN_W,STTN_H),interpolation=cv2.INTER_NEAREST) for f in frames_rgb]
    pil_masks  = [pil.fromarray((m*255).astype(np.uint8)) for m in masks_np]
    vlen = len(pil_frames)
    feats = _sttn_to_tensors(pil_frames).unsqueeze(0)*2-1
    masks = _sttn_to_tensors(pil_masks).unsqueeze(0)
    feats, masks = feats.to(DEVICE), masks.to(DEVICE)
    bin_masks = [np.expand_dims((np.array(m)!=0).astype(np.uint8),2) for m in pil_masks]
    raw_frames = [np.array(f).astype(np.uint8) for f in pil_frames]
    feats = sttn_model.encoder((feats*(1-masks).float()).view(vlen,3,STTN_H,STTN_W))
    _, c, fh, fw = feats.size(); feats = feats.view(1, vlen, c, fh, fw)
    comp = [None]*vlen
    for f in range(0, vlen, STTN_STRIDE):
        nb_ids = [i for i in range(max(0,f-STTN_STRIDE), min(vlen,f+STTN_STRIDE+1))]
        ref_ids = _sttn_get_ref_index(nb_ids, vlen)
        pred_feat = sttn_model.infer(feats[0, nb_ids+ref_ids], masks[0, nb_ids+ref_ids])
        pred_img = torch.tanh(sttn_model.decoder(pred_feat[:len(nb_ids)]))
        pred_img = ((pred_img+1)/2).cpu().permute(0,2,3,1).numpy()*255
        for i, idx in enumerate(nb_ids):
            im = pred_img[i].astype(np.uint8)*bin_masks[idx] + raw_frames[idx]*(1-bin_masks[idx])
            comp[idx] = im if comp[idx] is None else (comp[idx]*0.5 + im*0.5).astype(np.uint8)
    return [cv2.resize(c, (W0, H0), interpolation=cv2.INTER_LANCZOS4) for c in comp]

_STTN_CACHE = {}
def correct_endo_sttn(img_rgb, ds=None, kf=None, fid=None):
    if not _sttn_ok: return img_rgb
    if ds is None: return endo_sttn_inpaint_sequence([img_rgb])[0]
    key = (ds, kf)
    if key not in _STTN_CACHE:
        fids = SPLIT_BY_KF[key]
        seq = [load_split_frame(ds, kf, f)[0] for f in fids]
        out = endo_sttn_inpaint_sequence(seq)
        _STTN_CACHE.clear(); _STTN_CACHE[key] = {f:o for f,o in zip(fids, out)}
    return _STTN_CACHE[key][fid]

if _sttn_ok:
    CORRECTIONS["endosttn"] = correct_endo_sttn
print("Enhancements:", list(CORRECTIONS.keys()))

## 2. De la profundidad a la nube de puntos 3D

La **retroproyección** convierte cada píxel $(u,v)$ con profundidad $Z$ en un punto 3D usando los intrínsecos de la cámara $(f_x, f_y, c_x, c_y)$:

$$X = \frac{(u - c_x)\,Z}{f_x}, \qquad Y = \frac{(v - c_y)\,Z}{f_y}, \qquad Z = Z$$

Usamos los intrínsecos de SCARED (mitad izquierda, 1280×1024), los mismos del cálculo de Chamfer del Avance 5. La predicción de MonoIIT es relativa, así que se escala a milímetros con **median scaling** contra el *ground truth* (estándar en *depth* monocular) para que la nube esté en escala real y sea comparable con la del sensor.

In [ ]:
import numpy as np

# Intrinsecos de SCARED (camara izquierda, 1280x1024) — mismos que el Avance 5
FX, FY, CX, CY = 1078.0, 1078.0, 640.0, 512.0

def depth_to_pointcloud(depth_mm, rgb=None, mask=None, stride=2):
    """Retroproyecta un mapa de profundidad (mm) a nube de puntos 3D (Nx3) + color (Nx3).
    stride submuestrea para aligerar la visualizacion (1 = todos los pixeles)."""
    H, W = depth_mm.shape
    uu, vv = np.meshgrid(np.arange(W), np.arange(H))
    if mask is None:
        mask = np.isfinite(depth_mm) & (depth_mm > 0)
    if stride > 1:
        sub = np.zeros_like(mask); sub[::stride, ::stride] = True
        mask = mask & sub
    Z = depth_mm[mask]
    X = (uu[mask] - CX) * Z / FX
    Y = (vv[mask] - CY) * Z / FY
    pts = np.stack([X, Y, Z], axis=1)
    cols = rgb[mask] if rgb is not None else None
    return pts, cols

def median_scale(pred, gt, cap_mm=150.0):
    """Escala la prediccion relativa a mm con la mediana del GT valido (como en evaluate_depth)."""
    valid = np.isfinite(gt) & (gt > 0) & (gt < cap_mm)
    ratio = np.median(gt[valid]) / np.median(pred[valid])
    return pred * ratio, valid

print("Intrinsecos:", FX, FY, CX, CY)
print("Funciones de reconstruccion 3D listas.")

## 3. Reconstruir un fotograma

Elegimos un fotograma del *split* y reconstruimos **tres nubes** para compararlas:
1. **Ground truth** — la geometría real del sensor SCARED.
2. **MonoIIT (none)** — predicción sobre la imagen sin realce.
3. **MonoIIT (mejor enhancement)** — predicción sobre la imagen realzada (IAT por defecto, el realce más robusto del estudio).

In [ ]:
# --- Elegir un fotograma representativo (el mas brillante = caso interesante) ---
def _brillo(img): return cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0].mean()
_cands = [(_brillo(load_split_frame(ds,kf,f)[0]), ds, kf, f)
          for ds,kf,f in SPLIT_ITEMS if load_split_frame(ds,kf,f)[0] is not None]
_cands.sort()
_b, DS, KF, FID = _cands[-1]   # el mas brillante; cambia el indice para otro caso
print(f"Frame elegido: {DS}/{KF} frame {FID}  (L={_b:.0f})")

ENH_3D = "iat" if "iat" in CORRECTIONS else "none"   # mejor enhancement del estudio
STRIDE = 2   # submuestreo para visualizacion (sube a 3-4 si va lento)

img, gt_mm = load_split_frame(DS, KF, FID)
assert gt_mm is not None, "Ese frame no tiene ground truth; elige otro"

# Profundidad: baseline (none) y con enhancement
depth_none = predict_depth(img)
img_enh    = CORRECTIONS[ENH_3D](img, ds=DS, kf=KF, fid=FID) if ENH_3D=="endosttn" else CORRECTIONS[ENH_3D](img)
depth_enh  = predict_depth(img_enh)

# Median scaling a mm (cada prediccion contra el GT)
depth_none_mm, valid = median_scale(depth_none, gt_mm)
depth_enh_mm,  _     = median_scale(depth_enh,  gt_mm)

# Nubes de puntos
pc_gt,   col_gt   = depth_to_pointcloud(np.where(valid, gt_mm, np.nan),       rgb=img,     mask=valid, stride=STRIDE)
pc_none, col_none = depth_to_pointcloud(np.where(valid, depth_none_mm, np.nan), rgb=img,     mask=valid, stride=STRIDE)
pc_enh,  col_enh  = depth_to_pointcloud(np.where(valid, depth_enh_mm, np.nan),  rgb=img_enh, mask=valid, stride=STRIDE)
print(f"Puntos -> GT: {len(pc_gt)} | none: {len(pc_none)} | {ENH_3D}: {len(pc_enh)}")

## 4. Visualización 3D interactiva (Plotly)

Las tres nubes en un visor rotable. Cada punto lleva su **color RGB** original, de modo que la nube se ve como el tejido reconstruido en 3D. Rota, haz zoom y compara la forma de la predicción contra el ground truth.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import sys

# Renderer: en Colab usa "colab"; si aun asi no se ve inline, el HTML guardado SIEMPRE funciona.
try:
    if "google.colab" in sys.modules:
        pio.renderers.default = "colab"
except Exception:
    pass

# Sanity check: avisar si alguna nube esta vacia
for _n, _pc in [("GT", pc_gt), ("none", pc_none), (ENH_3D, pc_enh)]:
    if len(_pc) == 0:
        print(f"  AVISO: la nube '{_n}' esta VACIA -> revisa la celda anterior (mask/GT)")

def _rgb01(cols):
    # Plotly Scatter3d acepta color como array Nx3 normalizado a [0,1].
    return (cols.astype(float) / 255.0) if cols is not None else "#3b6fd4"

def _scatter(pc, cols, name):
    return go.Scatter3d(
        x=pc[:, 0], y=pc[:, 1], z=pc[:, 2], mode="markers", name=name,
        marker=dict(size=1.6, color=_rgb01(cols), opacity=0.9))

fig = make_subplots(rows=1, cols=3, specs=[[{"type": "scatter3d"}] * 3],
                    subplot_titles=("Ground truth", "MonoIIT (none)", f"MonoIIT ({ENH_3D})"),
                    horizontal_spacing=0.02)
fig.add_trace(_scatter(pc_gt,   col_gt,   "GT"),   row=1, col=1)
fig.add_trace(_scatter(pc_none, col_none, "none"), row=1, col=2)
fig.add_trace(_scatter(pc_enh,  col_enh,  ENH_3D), row=1, col=3)

_scene = dict(xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
              aspectmode="data", camera=dict(eye=dict(x=1.5, y=-1.5, z=0.9)))
fig.update_layout(height=560, width=1300, showlegend=False,
                  title_text=f"Reconstruccion 3D - {DS}/{KF} frame {FID}",
                  scene=_scene, scene2=_scene, scene3=_scene,
                  margin=dict(l=0, r=0, t=50, b=0))
fig.show()

# --- Guardar SIEMPRE el HTML interactivo (funciona aunque no se vea inline) ---
import os
_out = REPO_ROOT / "docs" / "reports" / "avance7_reconstruccion3d.html"
os.makedirs(_out.parent, exist_ok=True)
fig.write_html(str(_out), include_plotlyjs="cdn")
print(f"HTML 3D guardado en (ruta del KERNEL): {_out}")
print("  -> En Colab esta en /content/repo_52/... ; commitea o descarga ese archivo para abrirlo.")


## 5. Lectura

- **Ground truth** muestra la geometría real, pero **disperso** (el sensor estéreo deja huecos).
- **MonoIIT (none)** entrega una superficie **densa** y suave; la comparación cualitativa con el GT indica qué tan bien recupera la forma global del tejido.
- **MonoIIT (enhancement)** permite ver si el realce **cambia la geometría** reconstruida (idealmente la mejora o la deja estable, sin deformarla).

> **Nota.** La predicción monocular recupera la *forma relativa*; el *median scaling* la lleva a milímetros. Esta reconstrucción es por-fotograma (no fusiona varios frames), coherente con el alcance del estudio. La métrica cuantitativa que acompaña a esta vista es la **Chamfer Distance** entre nube predicha y GT (celda opcional del Avance 5).